# 🧩 TinyRecursiveModels - Local Evaluation & Test-Time Adaptation (TTA)

This notebook is a **restructured, local-friendly version** of the official Kaggle `arc2-trm-v31.ipynb` notebook. It has been stripped of Kaggle paths, environment-specific symlink hacks, and fine-tuned for robust local execution.

### Key Improvements Built-in:
1. **No Symlinks Needed:** Runs directly in your workspace repository root.
2. **Prefix-Robust Checkpoint Loading:** Strips compiled prefixes (`_orig_mod.`) dynamically to prevent runtime mismatches.
3. **Dynamic Puzzle Embedding Resizing:** Automatically matches and mean-initializes puzzle embedding dimensions if the evaluation dataset contains a different number of puzzles than the pre-training checkpoint.
4. **No strict check constraints:** Safely ignores unmatched parameters for seamless plug-and-play checkpoint evaluation.

## 1. Optional: Install Local Dependencies
Run this cell if you need to install required packages inside your local environment.

In [ ]:
# ! pip install hydra-core==1.3.2 adam_atan2_pytorch==0.2.4 argdantic==1.3.3 coolname==2.2.0 tqdm pydantic omegaconf

## 2. Generate Augmented Local Dataset
This step runs the dataset builder to compile the augmented inputs, labels, and puzzle indices. Set `--num-aug` (e.g. `128` or `1000` for paper-alignment).

In [ ]:
# Adjust data paths and num-aug as needed
! python -m dataset.build_arc_dataset \
  --input-file-prefix ./data1/arc-agi \
  --output-dir ./data1/arc2test-aug-128 \
  --subsets test \
  --test-set-name test \
  --num-aug 128

## 3. Run Joint Adaptation & Evaluation (TTA)
Run the robust local evaluation pipeline using `torchrun`. You can easily swap your checkpoint path, dataset path, epochs, and other hyperparameters here.

In [ ]:
# Standard local evaluation execution. Adjust checkpoint and data paths below.
! torchrun --standalone --nnodes=1 --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 \
  eval-arc-local.py \
  arch=trm \
  data_paths="[./data1/arc2test-aug-128]" \
  arch.L_layers=2 \
  arch.H_cycles=4 arch.L_cycles=4 arch.halt_max_steps=10 \
  freeze_weights=False \
  +load_checkpoint=./step_723914 \
  +checkpoint_path=./eval_checkpoint \
  eval_interval=4000 \
  epochs=4000 \
  global_batch_size=32 \
  ema=True \
  lr_warmup_steps=200 \
  lr=0.0001

## 4. Local Score Validation & Submission Parsing
This cell loads your generated `submission.json` and evaluates the exact match accuracy locally.

In [ ]:
import os
import json
import numpy as np

submission_file = "./eval_checkpoint/evaluator_SubmissionEvaluator_step_1200000/submission.json"
if not os.path.exists(submission_file):
    import glob
    sub_dirs = glob.glob("eval_checkpoint*/evaluator*/submission.json")
    if sub_dirs:
        submission_file = sub_dirs[0]
        print(f"Found submission file dynamically at: {submission_file}")

if os.path.exists(submission_file):
    with open(submission_file, "r") as f:
        submission = json.load(f)
    print(f"Loaded submission file containing {len(submission)} puzzles.")

    truth_file = "./data1/arc-agi_evaluation_solutions.json"
    if os.path.exists(truth_file):
        with open(truth_file, "r") as f:
            truth = json.load(f)
        
        scores = []
        for puzzle_name, test_attempts in submission.items():
            if puzzle_name in truth:
                puzzle_solution = truth[puzzle_name]
                puzzle_score = 0
                for tid, test_attempt in enumerate(test_attempts):
                    if tid < len(puzzle_solution):
                        sol = puzzle_solution[tid]
                        if test_attempt["attempt_1"] == sol or test_attempt["attempt_2"] == sol:
                            puzzle_score += 1
                scores.append(puzzle_score / len(test_attempts))
        
        if scores:
            print(f"Exact Match Accuracy: {sum(scores) / len(scores) * 100:.2f}%")
        else:
            print("No matching solution keys found in solution file.")
else:
    print("Submission file not found yet. Please run step 3 first!")